# Simplified LLM Function-Calling Agent Loop

This notebook implements the function-calling loop pattern from the attached instructions. It defines Python tools, describes them with JSON Schema, lets the LLM return structured `tool_calls`, executes the selected tool, stores the result in memory, and continues until the model calls `terminate`.

Before calling a real model, install LiteLLM and configure an API key:

```bash
python3 -m pip install litellm
export OPENAI_API_KEY=...
```

The default model is `openai/gpt-4o`.


In [1]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))

from function_agent.llm_agent import LLMFunctionCallingAgent

agent = LLMFunctionCallingAgent(root=repo_root, model="openai/gpt-4o")
print(json.dumps(agent.tools(), indent=2))


[
  {
    "type": "function",
    "function": {
      "name": "list_workspace_files",
      "description": "Returns files and directories from the workspace root. Use this before reading a workspace file so file names are accurate.",
      "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": false
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "read_workspace_text_file",
      "description": "Reads one UTF-8 text file under the workspace root. Use a file name or relative path returned by list_workspace_files.",
      "parameters": {
        "type": "object",
        "properties": {
          "file_name": {
            "type": "string",
            "description": "File name or relative path to read."
          }
        },
        "required": [
          "file_name"
        ],
        "additionalProperties": false
      }
    }
  },
  {
    "type": "function",
    "function": {
      "

In [2]:
user_task = input("What would you like me to do? ")

try:
    response = agent.run_loop(user_task, max_iterations=10)
    print(response.format())
except Exception as exc:
    print(f"LLM function-calling request failed: {type(exc).__name__}: {exc}")
    print("Install LiteLLM, set OPENAI_API_KEY, and confirm your account has available quota before running this cell against a real model.")


What would you like me to do?  list all files in the directory



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

LLM function-calling request failed: RateLimitError: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Install LiteLLM, set OPENAI_API_KEY, and confirm your account has available quota before running this cell against a real model.


## Optional: Validate Without API Credits

This cell uses a fake completion function to simulate the model choosing `list_workspace_files` and then `terminate`. It exercises the same agent loop without making a network call.


In [3]:
fake_responses = [
    {
        "choices": [
            {
                "message": {
                    "tool_calls": [
                        {
                            "function": {
                                "name": "list_workspace_files",
                                "arguments": "{}",
                            }
                        }
                    ]
                }
            }
        ]
    },
    {
        "choices": [
            {
                "message": {
                    "tool_calls": [
                        {
                            "function": {
                                "name": "terminate",
                                "arguments": json.dumps({"message": "Listed the workspace files."}),
                            }
                        }
                    ]
                }
            }
        ]
    },
]

fake_calls = []

def fake_completion(**kwargs):
    fake_calls.append(kwargs)
    return fake_responses[len(fake_calls) - 1]

demo_agent = LLMFunctionCallingAgent(root=repo_root, completion_fn=fake_completion)
demo_response = demo_agent.run_loop("tell me the files in the current directory")
print(demo_response.format())


Iteration 1
Tool Name: list_workspace_files
Tool Arguments: {}
Result:
{
  "files": [
    ".DS_Store",
    ".git/",
    ".githooks/",
    ".github/",
    ".gitignore",
    "app.py",
    "doc_agent.py",
    "docs/",
    "function_agent.py",
    "llm_function_agent.py",
    "notebooks/",
    "payments.db",
    "pyproject.toml",
    "README.md",
    "scripts/",
    "src/",
    "tests/"
  ]
}

Iteration 2
Tool Name: terminate
Tool Arguments: {'message': 'Listed the workspace files.'}
Result:
{
  "message": "Listed the workspace files."
}

Final: Listed the workspace files.
Stopped Reason: terminated
